# Введение

В этом дз мы поработаем с полезными базовыми возможностями langchain, которые обсуждались на лекции, \
а также, используя langchain, соберем простого бота Math Tutor — мини-бота, который отличает обычные вопросы от математических и умеет решать математические задачи через отдельный инструмент (`tool`).

# Задание 1

## Используя пример с лекции, создайте модель, промпт и LangChain-цепь

Настройте подключение к работающему inference-серверу. Для OpenAI-совместимого сервера (например, vLLM) используйте `ChatOpenAI`, для Ollama — `ChatOllama`. Перед запуском укажите параметры подключения вместо `...`.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Выберите нужный класс, не используйте выражение ChatOpenAI or ChatOllama:
model_class = ChatOpenAI  # либо ChatOllama
llm = model_class(
    ...  # TODO: укажите model, base_url/api_key для ChatOpenAI или model для ChatOllama
)

In [ ]:
HUMAN_SYSTEM_PROMPT = """
You are a considerate human who interacts with a user. Imagine that the user is a cat.

Rules:
- If the cat wants to eat, provide it with {food}.
- If the cat wants to play, reply briefly about playing with a fluffy ball, a laser pointer, or other cat activities.
- If the cat wants any other activity, respond that you can't understand what the cat wants.
"""


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate([
    ("system", HUMAN_SYSTEM_PROMPT),
    MessagesPlaceholder("messages")
])
chain = prompt_template | llm
chain.invoke({"messages": ["Мяу. Хочу есть"], 'food': 'salmon'})

## streaming

Запустите уже написанный chain в стриминге, дописав следующую функцию:

In [ ]:
def run_chain_with_streaming(chain, inputs):
    """Запускает chain в steaming режиме и принтит чанк на каждой итерации"""
    ...

## batching

Запустите уже написанный chain по батчам, дописав следующую функцию:

In [ ]:
def run_chain_with_batching(chain, inputs):
    """Запускает chain по батчам"""
    ...

## config

Сконфигурируйте температуру модели (через `configurable_fields` или параметры модели) и задайте `max_concurrency` в `RunnableConfig` для вызова `batch`. Учтите, что `max_concurrency` ограничивает параллелизм, а не размер батча.

In [ ]:
from langchain_core.runnables import ConfigurableField

def run_chain_with_config(chain, inputs, config):
    """Запускает chain с конфигом"""
    ...

## callbacks

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages.utils import convert_to_openai_messages

Реализуйте какой-нибудь callback для модели, например, `on_chat_model_start` (опционально, вы можете реализовать любой другой)

In [ ]:
class PrintStartHandler(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        body = {
            "messages": ...,
            "model": ...,
            "temperature": ...,
            # другие необходимые параметры из `extra_body` у модели
        }
        print(...)

In [ ]:
inputs = ...
config = {"callbacks": [PrintStartHandler()]}

In [ ]:
run_chain_with_config(chain=chain, inputs=inputs, config=config)

## Retries

In [ ]:
from langchain_core.runnables import Runnable, RunnableLambda

def timeout_until_attempt(k: int, chain: Runnable) -> RunnableLambda:
    """Бросает TimeoutError на первых k вызовах, затем вызывает chain."""
    state = {"n": 0}

    def gate(inputs):
        state["n"] += 1
        if state["n"] <= k:
            raise TimeoutError(f"timed out on attempt {state['n']}")
        return chain.invoke(inputs)

    return RunnableLambda(gate)

# Исправлено: в обёртку необходимо передать исходную цепочку.
primary_with_retry = timeout_until_attempt(k=2, chain=chain)

# TODO: вызовите primary_with_retry с with_retry таким образом, чтобы
# первые две попытки бросали TimeoutError, а третья выполнила chain.
# Полезные аргументы: retry_if_exception_type, stop_after_attempt,
# wait_exponential_jitter (метод Runnable.with_retry отличается от tenacity.retry).
inputs = {"messages": ["Мяу. Хочу есть"], "food": "salmon"}
retrying = primary_with_retry.with_retry(
    ...  # TODO: параметры with_retry
)
print(retrying.invoke(inputs))

# Проверка случая, когда попыток недостаточно:
will_fail = timeout_until_attempt(k=3, chain=chain).with_retry(
    retry_if_exception_type=(TimeoutError,),
    stop_after_attempt=2,
    wait_exponential_jitter=False,
)
try:
    will_fail.invoke(inputs)
except TimeoutError as e:
    print("still failing:", e)

# Задание 2: MathTutor chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

**Задание**:\
Далее вам нужно будет построить простого `MathTutor` на LangChain, используя `LCEL`.

**Принцип работы:**
- Если вопрос НЕ математический -> бот отвечает фразой что-то типа "Вам следует прислать вопрос по математике" (это делает первая llm)
- Если вопрос математический -> бот вызывает тул `solve_math_problem` (тул представляет собой вторую llm),
  получает решение и возвращает его пользователю.

1. Подготавливаем модели:

In [ ]:
# Для демонстрации можно использовать один и тот же сервер и модель.
# Выберите ChatOpenAI (OpenAI-совместимый API) либо ChatOllama (Ollama).
model_class = ChatOpenAI  # либо ChatOllama

# В качестве математической модели можно использовать Qwen2.5-Math.
tutor_llm = model_class(
    ...  # TODO: настройте подключение
)

math_solver_llm = model_class(
    ...  # TODO: настройте подключение
)

2. Сделайте фабрику инструмента на основе второй LLM. **Декоратор `@tool` должен стоять над внутренней функцией** `solve_math_problem`, а не над фабрикой `make_solve_math_problem_tool`. Добавьте понятное описание инструмента и его аргумента.

In [ ]:
def make_solve_math_problem_tool(model):
    @tool  # TODO: при желании задайте имя/схему аргументов
    def solve_math_problem(math_problem: str) -> str:
        """Решает математическую задачу и возвращает пошаговое решение с ответом."""
        # На вход получает полную формулировку задачи на русском или английском.
        # TODO:
        #   1. Задайте системный промпт для математического решения.
        #   2. Вызовите переданную модель model.invoke(...).
        #   3. Верните строку с content ответа (не объект AIMessage).
        # NOTE: по желанию используйте structured output для формата решения.
        ...
        result = model.invoke(...)
        return result.content

    return solve_math_problem

solve_math_problem_tool = make_solve_math_problem_tool(math_solver_llm)
tools = [solve_math_problem_tool]

3. Создадим промпт для LLM-tutor'а по следубщим правилам:
    1) Если сообщение пользователя НЕ математическая задача:
       - НЕ вызывать инструменты.
       - Ответить строго одной фразой по типу "Пожалуйста, пришлите математическую задачу"
   2) Если сообщение -- математическая задача:
       - НЕ решать задачу самому.
       - Сделать вызов тула `solve_math_problem`
Для создания промпта используете `ChatPromptTemplate`:

In [ ]:
tutor_prompt = ...

4. Добавим тулы к `tutor_llm`

In [ ]:
tutor_with_tools = ...

5. Реализовать звено, которое будет вызывать тул, `exec_tools`; звено, которое будет возвращать ответ модели, если вызова тула не было, `return_plain`,
   и соответствующий предикат `has_tool_call`

In [ ]:
def exec_tools(ai_message: AIMessage):
    """Исполнить tool_call solve_math."""
    ...
    result = ...
    return {'answer': result}

def return_plain(ai_message: AIMessage):
    """Просто вернуть текст ответа tutor_llm, если вызова tool не было"""
    ...
    result = ...
    return {'answer': result}

def has_tool_call(ai_message: AIMessage) -> bool:
    """
    Если LLM вызвал инструмент -> True, иначе -- False.
    Если True, то вызывается звено `exec_tools`, иначе вызывается `return_plain`
    """
    ...
    return

6. Используя `RunnableBranch`, сделайте завершающее звено в цепи:

In [ ]:
exec_tools_runnable = RunnableLambda(exec_tools)
return_plain_runnable = RunnableLambda(return_plain)

branch = RunnableBranch(...)

7. Собираем все вместе:

In [ ]:
chain = tutor_prompt | tutor_with_tools | branch

8. Проверьте оба сценария. Важно не только содержание ответа, но и маршрутизация: для математического запроса должен вызываться `solve_math_problem`, для нематематического — нет. Можно проверить `tutor_with_tools.invoke(...).tool_calls` на обоих запросах (при детерминированной конфигурации модели) и итоговое `chain.invoke(...)`. 

Пример запросов: «Привет! Как дела?» и «Реши уравнение 2x + 3 = 7».

In [ ]:
# Интерактивную проверку запускайте после реализации TODO выше.
# В автоматическом прогоне ноутбука бесконечный цикл с input() нужно пропустить.
while True:
    user_input = input("User: ").strip()
    if user_input.lower() in {"q", "quit"}:
        break

    result = chain.invoke({"messages": user_input})
    print("Bot:", result["answer"])